# P1 - Exploracio De Tickets

Objectiu: entendre el corpus de tickets, detectar soroll docent i preparar una classificació usable abans de provar cap baseline.

## Ruta de treball

1. Llegir les columnes i el significat de cada camp.
2. Detectar soroll, duplicats o casos quasi repetits.
3. Mirar si les categories actuals semblen consistents.
4. Tancar una llista de preguntes abans de passar a classificació.

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path('..') / 'dades' / 'tickets_helpdesk.csv'
df = pd.read_csv(DATA)
df

## 1. Lectura inicial

Abans de modelar, convé saber si el dataset és xicotet, si hi ha categories desbalancejades i si el text sembla net o repetitiu.

In [ ]:
df.info()
df.isna().sum().rename('nulls')

In [ ]:
df['departament_sospitat'].value_counts(dropna=False).rename_axis('departament').to_frame('tickets')

## 2. Longitud i repetició del text

Esta mirada ajuda a detectar tickets massa curts, repeticions i soroll de redacció.

In [ ]:
df['text_length'] = df['text_ticket'].str.len()
df[['ticket_id', 'departament_sospitat', 'text_length']].sort_values('text_length', ascending=False).head(10)

In [ ]:
duplicates = df[df.duplicated(subset=['text_ticket'], keep=False)].sort_values('text_ticket')
duplicates if not duplicates.empty else 'No hi ha duplicats exactes, pero cal revisar quasi duplicats.'

## 3. Paraules guia per categoria

Sense cap model encara, ja podem veure si hi ha paraules que semblen orientar la classificació.

In [ ]:
keywords = {
    'identitat': ['contrasenya', 'login', 'blocked', 'mfa', 'codi'],
    'xarxes': ['vpn', 'certificat', '403', 'oficina'],
    'correu': ['correu', 'quota', 'imap', 'adjunts'],
    'aplicacions': ['aplicacio', 'modul', 'panell', 'informes'],
}

def count_keyword_hits(text, words):
    text = text.lower()
    return sum(word in text for word in words)

for label, words in keywords.items():
    df[f'hits_{label}'] = df['text_ticket'].apply(lambda text: count_keyword_hits(text, words))

df[['ticket_id', 'departament_sospitat', 'hits_identitat', 'hits_xarxes', 'hits_correu', 'hits_aplicacions']].head(12)

## 4. Quasi duplicats o casos molt semblants

En este projecte és important detectar si el dataset conté variants del mateix problema, perquè això pot falsejar la sensació de qualitat del model.

In [ ]:
suspicious_patterns = df[df['text_ticket'].str.contains('quota plena|error de certificat|403|MFA|blocked', case=False, regex=True)]
suspicious_patterns[['ticket_id', 'departament_sospitat', 'text_ticket']]

## 5. Conclusions abans de classificar

Tanca ací tres idees:

- quines categories semblen més clares
- quins casos poden confondre el model
- quines paraules o patrons podrien servir per a una baseline inicial